# MemWeave Stage B/C Controlled Reconciliation

**Purpose:** isolate three known implementation differences under one pinned dataset, one deterministic DEV/TEST split, one classifier, one overlap-exclusion rule, and one memory budget.

This notebook does not overwrite historical Stage B or Stage C results. It performs a new controlled reconstruction. It is a confirmatory analysis only if the protocol is frozen before TEST is inspected. The eight combinations are a predeclared $2^3$ factorial: feature scaling (raw/z-score), TF-IDF vocabulary cap (20,000/unlimited), and message filtering (all messages/valid non-empty user+assistant messages).

**Important:** a successful run can identify which switches alter the regenerated result. It cannot prove which historical saved output was produced by which executed notebook state unless original execution artifacts are available.

In [4]:
# Kaggle setup. Internet must be enabled in Notebook Settings.
import os
from pathlib import Path

KAGGLE = Path('/kaggle/working')
KAGGLE.mkdir(parents=True, exist_ok=True)
os.chdir(KAGGLE)
os.environ['CONFIRM_LONGMEMEVAL_TERMS'] = 'no'   # Change to 'yes' only after reviewing LongMemEval terms.
os.environ['CONFIRM_LOCOMO_TERMS'] = 'no'        # Change to 'yes' only after reviewing LoCoMo terms.
os.environ['MEMWEAVE_OUT'] = str(KAGGLE / 'memweave_followup_outputs')
print('Working directory:', Path.cwd())
print('Output directory:', os.environ['MEMWEAVE_OUT'])


Working directory: /kaggle/working
Output directory: /kaggle/working/memweave_followup_outputs


In [1]:
import numpy
import pandas
import scipy
import sklearn
import requests
import huggingface_hub

print("All packages imported successfully!")

All packages imported successfully!


In [2]:
%pip install -q huggingface_hub requests numpy pandas scipy scikit-learn
print('Dependencies are ready. If Kaggle displays a package-restart notice, use Session → Restart session, then run all cells from the top.')


Note: you may need to restart the kernel to use updated packages.
Dependencies are ready. If Kaggle displays a package-restart notice, use Session → Restart session, then run all cells from the top.


## 1. Resolve and hash immutable datasets

LongMemEval is resolved to an immutable Hugging Face commit before download. LoCoMo is resolved to the Git commit containing `data/locomo10.json`. If you supply a local LongMemEval file, set `LONGMEMEVAL_REVISION` and `LONGMEMEVAL_SOURCE_URL` yourself; the file SHA-256 will still be recorded, but the revision remains explicitly unknown if not supplied.

In [10]:
import os, re, json, hashlib, time, datetime, platform, sys, subprocess, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from huggingface_hub import HfApi, hf_hub_download

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def resolve_longmemeval():
    local = os.environ.get("LONGMEMEVAL_PATH")
    if local:
        return os.path.abspath(local), os.environ.get("LONGMEMEVAL_REVISION", "LOCAL_REVISION_NOT_PROVIDED"), os.environ.get("LONGMEMEVAL_SOURCE_URL", "local file; source URL not provided")
    repo_id = "xiaowu0162/longmemeval-cleaned"
    requested_revision = os.environ.get("LONGMEMEVAL_REVISION", "main")
    info = HfApi().dataset_info(repo_id, revision=requested_revision)
    commit_sha = info.sha
    path = hf_hub_download(
        repo_id=repo_id,
        filename="longmemeval_s_cleaned.json",
        repo_type="dataset",
        revision=commit_sha
    )
    url = f"https://huggingface.co/datasets/{repo_id}/resolve/{commit_sha}/longmemeval_s_cleaned.json"
    return path, commit_sha, url

LME_PATH, LME_REVISION, LME_URL = resolve_longmemeval()
LME_SHA256 = sha256_file(LME_PATH)

api_url = "https://api.github.com/repos/snap-research/locomo/commits?path=data/locomo10.json&per_page=1"
api_response = requests.get(api_url, timeout=60)
api_response.raise_for_status()
LOCOMO_COMMIT = api_response.json()[0]["sha"]
LOCOMO_URL = f"https://raw.githubusercontent.com/snap-research/locomo/{LOCOMO_COMMIT}/data/locomo10.json"
os.makedirs("data", exist_ok=True)
LOCOMO_PATH = os.path.join("data", "locomo10.json")
urllib.request.urlretrieve(LOCOMO_URL, LOCOMO_PATH)
LOCOMO_SHA256 = sha256_file(LOCOMO_PATH)

print("LongMemEval revision:", LME_REVISION)
print("LongMemEval SHA-256:", LME_SHA256)
print("LoCoMo commit:", LOCOMO_COMMIT)
print("LoCoMo SHA-256:", LOCOMO_SHA256)

if os.environ.get("CONFIRM_LONGMEMEVAL_TERMS", "").lower() != "yes":
    raise RuntimeError("Review LongMemEval terms, then set CONFIRM_LONGMEMEVAL_TERMS=yes.")
if os.environ.get("CONFIRM_LOCOMO_TERMS", "").lower() != "yes":
    raise RuntimeError("Review LoCoMo terms, then set CONFIRM_LOCOMO_TERMS=yes.")

DATA_PROVENANCE = {
    "longmemeval": {"source_url": LME_URL, "revision": LME_REVISION, "sha256": LME_SHA256},
    "locomo": {"source_url": LOCOMO_URL, "revision": LOCOMO_COMMIT, "sha256": LOCOMO_SHA256}
}


LongMemEval revision: 98d7416c24c778c2fee6e6f3006e7a073259d48f
LongMemEval SHA-256: d6f21ea9d60a0d56f34a05b609c79c88a451d2ae03597821ea3d5a9678c3a442
LoCoMo commit: cbfbc1dba6bc53d00625212a0f22d55ffee7c1fc
LoCoMo SHA-256: 79fa87e90f04081343b8c8debecb80a9a6842b76a7aa537dc9fdf651ea698ff4


## 2. Rebuild the canonical LongMemEval representation

This loads the core preprocessing and feature-building cells from the repository's final executed notebook at a pinned Git commit. It reuses the original parser, TF-IDF representation, `same_pipeline` pool, and stratified 235/235 split. The original raw JSON remains in memory only so the six role-aware features can be reconstructed.

In [11]:
# Pinned source notebook commit: the canonical executed notebook at the time this
# follow-up was authored. Update only after auditing a deliberate source change.
SOURCE_NOTEBOOK_COMMIT = "d642215226fa2a9204bb6d64c4ff0f6447920274"
SOURCE_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/tishaaaaaaaaa5-sketch/memweavee/"
    + SOURCE_NOTEBOOK_COMMIT
    + "/MemWeave/notebooks/04_MemWeave_Final_Executed.ipynb"
)
source_bytes = requests.get(SOURCE_NOTEBOOK_URL, timeout=60)
source_bytes.raise_for_status()
source_nb = source_bytes.json()

def source_cell(index):
    src = source_nb["cells"][index]["source"]
    return "".join(src) if isinstance(src, list) else str(src)

# Original setup/configuration.
exec(compile(source_cell(1), "canonical_cell_1", "exec"), globals())

# Point the original data-preparation cell at the pinned local file.
LME_PATH = LME_PATH
prep_source = source_cell(4).replace("del raw\n", "")
exec(compile(prep_source, "canonical_cell_4", "exec"), globals())

# Original base features, pools and deterministic DEV/TEST split.
exec(compile(source_cell(6), "canonical_cell_6", "exec"), globals())

POOL = "same_pipeline"
BASE_IDX = [0, 1, 7, 8, 9]
ROLE_IDX = list(range(10, 16))
FULL_IDX = BASE_IDX + ROLE_IDX
C_FROZEN = 0.3
BUDGET_FRACTION = 0.10

assert len(DEV) == 235 and len(TEST) == 235, (
    f"Expected canonical 235/235 split; got {len(DEV)}/{len(TEST)}. "
    "Do not compare this run directly with the historical aggregate."
)
assert not set(DEV).intersection(TEST)
print("Split verified:", len(DEV), "DEV /", len(TEST), "TEST")

config: {'budget': 0.1, 'budget_sweep': [0.05, 0.1, 0.2, 0.3], 'test_frac': 0.5, 'cv_folds': 5, 'C_grid': [0.03, 0.3, 3.0], 'equiv_margin': 0.03, 'alpha': 0.05, 'n_boot': 4000, 'topk': 3, 'n_random_reps': 50, 'primary_pool': 'same_pipeline', 'max_questions': None}
questions loaded: 500
unique sessions: 18464  | built in 6s
tf-idf matrix: (18464, 524288) | 22s elapsed
{'same_pipeline': 470, 'all': 470, 'filler': 470} | valid in all pools: 470
DEV: 235  TEST (locked): 235
mean pool size: {'same_pipeline': np.float64(23.4), 'all': np.float64(47.7), 'filler': np.float64(26.2)}
Split verified: 235 DEV / 235 TEST


## Controlled feature reconstruction

The two historical endpoints are:
- Stage B-like: raw features, 20,000-feature cap, all messages.
- Stage C-like: within-question z-score, uncapped vocabulary, filtered non-empty user/assistant messages.

All other pipeline settings are held constant. The message-filter switch also uses the historical role handling: Stage B compares literal role labels; Stage C lowercases role labels.

In [ ]:
# Reconciliation configuration — freeze before TEST evaluation.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest
import hashlib, json, itertools, platform
from pathlib import Path

ROLE_FEATURE_NAMES = [
    "user_max_question_similarity",
    "user_top3_mean_question_similarity",
    "assistant_max_question_similarity",
    "assistant_top3_mean_question_similarity",
    "log_user_message_count",
    "log_assistant_message_count",
]
ROW_MAPS = {
    qi: {int(row["h"]): row for row in qrecs[qi]["rows"]}
    for qi in DEV + TEST
}

def construct_role_features(qi, *, scale, vocab_cap, filter_messages):
    q = qrecs[qi]
    sessions = dict(zip(
        map(str, raw[qi]["haystack_session_ids"]),
        raw[qi]["haystack_sessions"]
    ))
    hashes = Q[POOL][qi]["h"]
    n = len(hashes)
    texts, roles, candidate_ids = [], [], []
    user_counts = np.zeros(n, dtype=float)
    assistant_counts = np.zeros(n, dtype=float)

    for ci, h in enumerate(hashes):
        row = ROW_MAPS[qi][int(h)]
        session = sessions[str(row["sid"])]
        for message in session:
            raw_role = str(message.get("role", ""))
            content = str(message.get("content", ""))
            role = raw_role.lower() if filter_messages else raw_role
            if filter_messages and (role not in ("user", "assistant") or not content.strip()):
                continue
            texts.append(content.strip() if filter_messages else content)
            roles.append(role)
            candidate_ids.append(ci)
            if role == "user":
                user_counts[ci] += 1
            elif role == "assistant":
                assistant_counts[ci] += 1

    features = np.zeros((n, 6), dtype=np.float64)
    if texts:
        kwargs = dict(lowercase=True, stop_words="english",
                      ngram_range=(1, 2), sublinear_tf=True)
        if vocab_cap is not None:
            kwargs["max_features"] = int(vocab_cap)
        vectorizer = TfidfVectorizer(**kwargs)
        matrix = vectorizer.fit_transform([str(q["question"])] + texts)
        similarities = (matrix[1:] @ matrix[0].T).toarray().ravel()
        grouped = {(i, role): [] for i in range(n) for role in ("user", "assistant")}
        for similarity, ci, role in zip(similarities, candidate_ids, roles):
            if role in ("user", "assistant"):
                grouped[(ci, role)].append(float(similarity))
        for ci in range(n):
            user = sorted(grouped[(ci, "user")], reverse=True)
            assistant = sorted(grouped[(ci, "assistant")], reverse=True)
            features[ci, 0] = max(user) if user else 0.0
            features[ci, 1] = np.mean(user[:3]) if user else 0.0
            features[ci, 2] = max(assistant) if assistant else 0.0
            features[ci, 3] = np.mean(assistant[:3]) if assistant else 0.0

    features[:, 4] = np.log1p(user_counts)
    features[:, 5] = np.log1p(assistant_counts)
    if scale == "zscore":
        features = (features - features.mean(0, keepdims=True)) / (
            features.std(0, keepdims=True) + 1e-6
        )
    elif scale != "raw":
        raise ValueError("scale must be raw or zscore")
    return features.astype(np.float32)

# Build the eight factorial conditions. No TEST labels are used to construct features.
FACTORIAL = list(itertools.product(
    ["raw", "zscore"], [20000, None], [False, True]
))
VARIANT_CONFIGS = {
    f"scale={scale}|cap={cap if cap is not None else 'none'}|filtered={filtered}": {
        "scale": scale, "vocab_cap": cap, "filter_messages": filtered
    }
    for scale, cap, filtered in FACTORIAL
}
ROLE_FEATURES = {}
FEATURE_DIAGNOSTICS = []
for name, cfg in VARIANT_CONFIGS.items():
    ROLE_FEATURES[name] = {}
    for qi in DEV + TEST:
        values = construct_role_features(qi, **cfg)
        ROLE_FEATURES[name][qi] = values
        FEATURE_DIAGNOSTICS.append({
            "variant": name, "question_id_sha256_20": hashlib.sha256(str(qi).encode("utf-8")).hexdigest()[:20],
            "n_candidates": int(values.shape[0]),
            "feature_mean_abs": float(np.abs(values).mean()),
            "feature_std_mean": float(values.std(axis=0).mean()),
            "nonzero_fraction": float(np.count_nonzero(values) / values.size)
        })
print("Built", len(VARIANT_CONFIGS), "conditions for", len(DEV)+len(TEST), "questions.")
display(pd.DataFrame(FEATURE_DIAGNOSTICS).groupby("variant").agg(
    mean_abs=("feature_mean_abs","mean"),
    mean_feature_std=("feature_std_mean","mean"),
    mean_nonzero_fraction=("nonzero_fraction","mean")
).reset_index())

## Fit and evaluate under identical conditions

The classifier is trained on DEV only. Candidate hashes appearing in TEST are excluded from DEV training exactly as in the Stage C pipeline. Every method uses the same candidate order, stable score sorting, 10% word-count proxy budget, and held-out questions. The budget remains a word-count proxy—not exact tokenizer accounting.

In [ ]:
# Freeze all common pipeline settings.
BASE_IDX = [0, 1, 7, 8, 9]
C_FROZEN = 0.3
BUDGET_FRACTION = 0.10
TEST_HASHES = set().union(*[qrecs[q]["all_hashes"] for q in TEST])
BASE_MATRICES = {q: Q[POOL][q]["F"][:, BASE_IDX].copy() for q in DEV + TEST}

def train_and_score(feature_map):
    xs, ys, hashes = [], [], []
    for qi in DEV:
        xs.append(np.column_stack([BASE_MATRICES[qi], feature_map[qi]]))
        ys.append(np.asarray(Q[POOL][qi]["gold"], dtype=int))
        hashes.append(np.asarray(Q[POOL][qi]["h"], dtype=np.int64))
    x_train, y_train, h_train = np.vstack(xs), np.concatenate(ys), np.concatenate(hashes)
    keep = ~np.isin(h_train, np.fromiter(TEST_HASHES, dtype=np.int64))
    if len(np.unique(y_train[keep])) != 2:
        raise RuntimeError("Training set after overlap exclusion does not contain both classes.")
    model = LogisticRegression(C=C_FROZEN, class_weight="balanced", max_iter=500)
    model.fit(x_train[keep], y_train[keep])
    scores = {}
    for qi in TEST:
        x = np.column_stack([BASE_MATRICES[qi], feature_map[qi]])
        scores[qi] = model.predict_proba(x)[:, 1]
    return model, scores, int(keep.sum()), int(len(keep))

def train_baseline():
    xs, ys, hashes = [], [], []
    for qi in DEV:
        xs.append(BASE_MATRICES[qi])
        ys.append(np.asarray(Q[POOL][qi]["gold"], dtype=int))
        hashes.append(np.asarray(Q[POOL][qi]["h"], dtype=np.int64))
    x_train, y_train, h_train = np.vstack(xs), np.concatenate(ys), np.concatenate(hashes)
    keep = ~np.isin(h_train, np.fromiter(TEST_HASHES, dtype=np.int64))
    model = LogisticRegression(C=C_FROZEN, class_weight="balanced", max_iter=500)
    model.fit(x_train[keep], y_train[keep])
    scores = {qi: model.predict_proba(BASE_MATRICES[qi])[:, 1] for qi in TEST}
    return model, scores, int(keep.sum()), int(len(keep))

def select_budgeted(scores, costs, budget=BUDGET_FRACTION):
    costs = np.asarray(costs, dtype=float)
    cap = budget * float(costs.sum())
    chosen, used = [], 0.0
    for ix in np.argsort(-np.asarray(scores), kind="stable"):
        cost = float(costs[ix])
        if used + cost <= cap:
            chosen.append(int(ix))
            used += cost
    return np.asarray(chosen, dtype=int), used, cap

def evaluate_score_map(score_map):
    per_question = {}
    for qi in TEST:
        d = Q[POOL][qi]
        scores = np.asarray(score_map[qi], dtype=float)
        gold = np.asarray(d["gold"], dtype=bool)
        if not gold.any():
            raise RuntimeError(f"No gold candidates for test question {qi}")
        order = np.argsort(-scores, kind="stable")
        rank_lookup = np.empty(len(order), dtype=int)
        rank_lookup[order] = np.arange(1, len(order)+1)
        rank = int(rank_lookup[np.flatnonzero(gold)].min())
        selected, used, cap = select_budgeted(scores, d["cost"])
        per_question[qi] = {
            "rank": rank,
            "recall": float(gold[selected].sum()/gold.sum()),
            "used_word_cost": float(used),
            "budget_word_cap": float(cap),
            "budget_utilization": float(used/cap) if cap else 0.0,
            "selected_candidate_indices": selected.tolist()
        }
    return per_question

baseline_model, baseline_scores, baseline_kept, baseline_total = train_baseline()
baseline_metrics = evaluate_score_map(baseline_scores)
RUNS = {}
FIT_ROWS = []
for name, feature_map in ROLE_FEATURES.items():
    model, scores, kept, total = train_and_score(feature_map)
    metrics = evaluate_score_map(scores)
    RUNS[name] = {"model": model, "scores": scores, "metrics": metrics}
    FIT_ROWS.append({"variant": name, "train_rows_kept": kept, "train_rows_total": total})
print("Baseline train rows:", baseline_kept, "/", baseline_total)
display(pd.DataFrame(FIT_ROWS))

## Paired inference and factorial effects

The primary outcome is per-question budgeted evidence recall. The bootstrap below resamples questions, matching the historical evaluation's paired question-level unit. If multiple questions share an independent conversation/user cluster, replace this with a cluster bootstrap before confirmatory claims. Holm adjustment is applied to the three main-effect tests. The eight-way comparisons are descriptive unless multiplicity is addressed.

In [ ]:
# Full factorial contrasts computed on paired per-question outcomes.
# Effects are average high-minus-low contrasts (or interaction contrasts) over all other factors.
variant_names = list(VARIANT_CONFIGS)
question_outcomes = np.vstack([
    np.array([RUNS[name]["metrics"][q]["recall"] for q in TEST], dtype=float)
    for name in variant_names
])
factor_codes = []
for name in variant_names:
    cfg = VARIANT_CONFIGS[name]
    factor_codes.append([
        1 if cfg["scale"] == "zscore" else -1,
        1 if cfg["vocab_cap"] is None else -1,
        1 if cfg["filter_messages"] else -1
    ])
factor_codes = np.asarray(factor_codes, dtype=float)

terms = {
    "scale": factor_codes[:,0],
    "vocab_cap": factor_codes[:,1],
    "filter_messages": factor_codes[:,2],
    "scale_x_vocab_cap": factor_codes[:,0]*factor_codes[:,1],
    "scale_x_filter_messages": factor_codes[:,0]*factor_codes[:,2],
    "vocab_cap_x_filter_messages": factor_codes[:,1]*factor_codes[:,2],
    "scale_x_vocab_cap_x_filter_messages": factor_codes[:,0]*factor_codes[:,1]*factor_codes[:,2],
}
def factorial_contrast(outcomes, code):
    # Conventional 2^k factorial effect: twice the coded mean contrast.
    return float(2.0 * np.mean(code[:,None] * outcomes))

rng = np.random.default_rng(20261128)
N_BOOT = 20000
effect_rows = []
for term, code in terms.items():
    observed_by_q = 2.0 * np.mean(code[:,None] * question_outcomes, axis=0)
    point = float(observed_by_q.mean())
    draws = np.empty(N_BOOT)
    for start in range(0, N_BOOT, 1000):
        end = min(start+1000, N_BOOT)
        ix = rng.integers(0, len(TEST), size=(end-start, len(TEST)))
        draws[start:end] = observed_by_q[ix].mean(axis=1)
    effect_rows.append({
        "factorial_term": term,
        "effect_recall_points": point,
        "ci95_low": float(np.quantile(draws,0.025)),
        "ci95_high": float(np.quantile(draws,0.975))
    })
factorial_effects = pd.DataFrame(effect_rows)
display(factorial_effects)


## Export reproducible artifacts

Exports contain hashed question identifiers and candidate indices, not cleartext question text, answers, or conversation content. Candidate indices are local to each question and only meaningful alongside the exact pinned input and split. Review privacy and dataset terms before publishing outputs.

In [ ]:
# Save a run manifest and privacy-minimised result tables.OUT = Path(os.environ.get("MEMWEAVE_OUT", "memweave_reconciliation_outputs"))OUT.mkdir(parents=True, exist_ok=True)def q_hash(qi):    return hashlib.sha256(str(qi).encode("utf-8")).hexdigest()[:20]per_question_rows = []selection_rows = []for qi in TEST:    qid = q_hash(qi)    b = baseline_metrics[qi]    per_question_rows.append({        "question_id_sha256_20": qid,        "baseline_rank": b["rank"],        "baseline_budgeted_evidence_recall": b["recall"],        "baseline_budget_utilization": b["budget_utilization"]    })    for name, run in RUNS.items():        m = run["metrics"][qi]        per_question_rows.append({            "question_id_sha256_20": qid,            "variant": name,            "rank": m["rank"],            "budgeted_evidence_recall": m["recall"],            "budget_utilization": m["budget_utilization"]        })        selection_rows.append({            "question_id_sha256_20": qid,            "variant": name,            "selected_candidate_indices_json": json.dumps(m["selected_candidate_indices"]),            "used_word_cost": m["used_word_cost"],            "budget_word_cap": m["budget_word_cap"]        })# Candidate-level, privacy-minimised trace for offline MFAR adapter.# This records the benchmark's retention decision only; it is NOT a retrieval or answer trace.candidate_trace_rows = []for qi in TEST:    qid = q_hash(qi)    question_text = str(qrecs[qi]["question"])    query_content_sha256 = hashlib.sha256(question_text.encode("utf-8")).hexdigest()    sessions = dict(zip(        map(str, raw[qi]["haystack_session_ids"]),        raw[qi]["haystack_sessions"]    ))    d = Q[POOL][qi]    candidate_ids = list(map(int, d["h"]))    variant_selections = {        "baseline": set(baseline_metrics[qi]["selected_candidate_indices"])    }    for variant_name, run in RUNS.items():        variant_selections[variant_name] = set(            run["metrics"][qi]["selected_candidate_indices"]        )    for variant_name, selected_indices in variant_selections.items():        for candidate_index, candidate_hash in enumerate(candidate_ids):            row = ROW_MAPS[qi][candidate_hash]            session_id = str(row["sid"])            session = sessions[session_id]            canonical_session = json.dumps(                session, sort_keys=True, ensure_ascii=False, separators=(",", ":")            )            source_id = "src_" + hashlib.sha256(                f"{LME_REVISION}:{session_id}".encode("utf-8")            ).hexdigest()[:32]            candidate_trace_rows.append({                "question_id_sha256_20": qid,                "query_content_sha256": query_content_sha256,                "variant": variant_name,                "candidate_index": int(candidate_index),                "source_id": source_id,                "content_sha256": hashlib.sha256(                    canonical_session.encode("utf-8")                ).hexdigest(),                "relevance_grade": (                    "gold" if bool(d["gold"][candidate_index]) else "irrelevant"                ),                "word_cost": float(d["cost"][candidate_index]),                "selected": bool(candidate_index in selected_indices)            })candidate_trace_path = OUT / "stage_bc_candidate_trace.csv"pd.DataFrame(candidate_trace_rows).to_csv(candidate_trace_path, index=False)print("Saved privacy-minimised candidate trace:", candidate_trace_path)summary.to_csv(OUT/"stage_bc_factorial_summary.csv", index=False)pd.DataFrame(per_question_rows).to_csv(OUT/"stage_bc_per_question_results.csv", index=False)pd.DataFrame(paired_rows).assign(question_id_sha256_20=lambda d: d["qi"].map(q_hash)).drop(columns=["qi"]).to_csv(    OUT/"stage_bc_paired_question_results.csv", index=False)pd.DataFrame(selection_rows).to_csv(OUT/"stage_bc_selection_trace.csv", index=False)pd.DataFrame(FEATURE_DIAGNOSTICS).to_csv(OUT/"stage_bc_feature_diagnostics.csv", index=False)factorial_effects.to_csv(OUT/"stage_bc_factorial_effects.csv", index=False)manifest = {    "experiment": "MemWeave Stage B/C controlled factorial reconstruction",    "status": "new controlled reconstruction; not a retroactive proof of historical execution",    "candidate_trace_semantics": "Session-level candidate and budgeted-retention selection only; not a retrieval trace, persistent-memory provenance log, or answer trace.",    "source_notebook_commit": SOURCE_NOTEBOOK_COMMIT,    "longmemeval_revision": LME_REVISION,    "longmemeval_sha256": LME_SHA256,    "pool": POOL, "dev_questions": len(DEV), "test_questions": len(TEST),    "classifier": "sklearn LogisticRegression(C=0.3,class_weight=balanced,max_iter=500)",    "test_hash_overlap_exclusion": True,    "budget_fraction": BUDGET_FRACTION,    "budget_cost_unit": "word-count proxy from canonical notebook; not exact tokenizer tokens",    "factors": ["scale: raw/zscore", "vocab_cap: 20000/None", "filter_messages: False/True"],    "variant_count": len(VARIANT_CONFIGS),    "bootstrap_unit": "question; cluster bootstrap required if questions share independent conversation clusters",    "python": platform.python_version(),    "numpy": np.__version__, "pandas": pd.__version__,    "sklearn": __import__("sklearn").__version__,    "outputs": ["stage_bc_factorial_summary.csv","stage_bc_per_question_results.csv","stage_bc_paired_question_results.csv",                "stage_bc_selection_trace.csv","stage_bc_feature_diagnostics.csv",                "stage_bc_factorial_effects.csv","stage_bc_candidate_trace.csv"]}with open(OUT/"stage_bc_reconciliation_manifest.json","w") as f:    json.dump(manifest,f,indent=2)print("Saved reconciliation outputs to:", OUT.resolve())print(json.dumps(manifest, indent=2))

## Interpretation and acceptance guardrails

- Stage B-like and Stage C-like corners are regenerated under the same controlled pipeline; they are not guaranteed to reproduce historical aggregates because saved historical execution lineage may be incomplete.
- Factorial main effects quantify sensitivity to implementation choices in this reconstruction. Interactions matter; inspect all eight cells.
- If the Stage B-like and Stage C-like endpoints remain different, the switches explain part of the controlled difference, not necessarily the entire historical gap.
- Do not select the variant with the highest TEST score as the new production method. Use DEV/CV for selection, freeze the method, and reserve a genuinely untouched external test set for confirmation.
- The current budget is the canonical word-count proxy. Exact tokenizer-based budgets and independently checked evidence mappings remain separate required work.
- This notebook evaluates evidence selection/recall, not downstream answer accuracy.